# Embed the Policy Summary

| Item | Description |
|---|---|
| Goal | Turn the policy summary of T3.2.3 into one unit-length embedding, the topic-side vector of the main relevance measure. |
| Input | `policy_summary.md` and `policy_summary_meta.json` from T3.2.3 in the T3.2 Processed folder. |
| Output | `policy_summary_embedding.npy` (1 × 1024, float32) and `policy_summary_text.csv` (the text with its token count) in the T3.2 Processed folder. |
| Model | `BAAI/bge-large-en-v1.5`: the CLS token of the last layer, L2-normalized, at most 512 input tokens, run on the GPU in float32. The same settings embed the firm summaries in T3.3.6. |
| Run | Rerun 2026-10-31 on one A40; the vector equals the one of the October 2026 run. |


## Step 1. Import packages


In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from IPython.display import display


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.2 - Topic Side - Policy Summary of the Legislator Statements")
NOTEBOOK_NAME = "T3.2.5 - Embed - The Policy Summary Vector"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
SUMMARY_FILE = PROCESSED_FOLDER / "policy_summary.md"
META_FILE = PROCESSED_FOLDER / "policy_summary_meta.json"
TOPIC_ID = 68  # consumer data privacy legislation
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (PROCESSED_FOLDER,):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Point the Hugging Face cache at the scratch root


In [3]:
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 4. Load the policy summary

The text and the token count T3.2.3 recorded for it.


In [4]:
policy_summary = SUMMARY_FILE.read_text(encoding="utf-8").strip()
meta = json.loads(META_FILE.read_text(encoding="utf-8"))
assert policy_summary and len(policy_summary.split()) == meta["final_n_words"]
print(f"Policy summary: {meta['final_n_words']} words, {meta['final_n_bge_tokens']} encoder tokens as recorded by T3.2.3\n")
print(policy_summary)


Policy summary: 221 words, 274 encoder tokens as recorded by T3.2.3

The legislation imposes rigorous obligations on tech giants, broadband providers, healthcare organizations, drone operators, and financial institutions to secure consumer data and prevent its unauthorized sale or misuse. It grants individuals specific rights, including the ability to opt in or out of data collection, demand clear and conspicuous privacy policies, control their digital communications, and establish data "firewalls." Companies are required to implement "privacy by design," provide breach notifications within 72 hours, and ensure that sensitive information such as health records, social security numbers, and geolocation data is protected from discrimination and identity theft.

The regulatory scope targets a wide array of data practices, ranging from the deceptive monitoring of online activities and the unauthorized mapping of consumer movements to the commercialization of private data and the use of alg

## Step 5. Load the encoder


In [5]:
ENCODER = "BAAI/bge-large-en-v1.5"  # the encoder of T1.3.2 and T2; CLS pooling is the pooling this checkpoint publishes
MAX_TOKENS = 512  # the encoder's input limit

tokenizer = AutoTokenizer.from_pretrained(ENCODER)
assert torch.cuda.is_available(), "run this notebook on a GPU node"
model = AutoModel.from_pretrained(ENCODER).to("cuda").eval()
assert model.config.max_position_embeddings == MAX_TOKENS
print(f"Encoder: {ENCODER}; hidden size {model.config.hidden_size}; input limit {MAX_TOKENS} tokens")
print(f"Device: {torch.cuda.get_device_name(0)}; dtype {next(model.parameters()).dtype}")


Encoder: BAAI/bge-large-en-v1.5; hidden size 1024; input limit 512 tokens
Device: NVIDIA A40; dtype torch.float32


## Step 6. Count the tokens and embed the text

The recount must equal the recorded count and fit the limit. The encoder runs once; the CLS token of the last layer is scaled to unit length.


In [6]:
n_tokens = len(tokenizer(policy_summary)["input_ids"])
assert n_tokens == meta["final_n_bge_tokens"] and n_tokens <= MAX_TOKENS
with torch.no_grad():
    encoded = tokenizer(policy_summary, truncation=True, max_length=MAX_TOKENS, return_tensors="pt").to("cuda")
    policy_summary_embedding = torch.nn.functional.normalize(model(**encoded).last_hidden_state[:, 0], dim=-1).cpu().numpy().astype(np.float32)
assert policy_summary_embedding.shape == (1, model.config.hidden_size) and abs(np.linalg.norm(policy_summary_embedding) - 1) < 1e-5
print(f"Tokens including the start and end tokens: {n_tokens}; shape {policy_summary_embedding.shape}; first five values {np.round(policy_summary_embedding[0, :5], 4)}")


Tokens including the start and end tokens: 274; shape (1, 1024); first five values [-0.0379 -0.0295  0.0027  0.039   0.0234]


## Step 7. Save the text and the vector

The text sits beside the vector with its token count, so a reader sees exactly what was embedded.


In [7]:
pd.DataFrame([{"topic_id": TOPIC_ID, "source_run_started": meta["run_started"], "policy_summary": policy_summary, "n_words": meta["final_n_words"], "n_tokens": n_tokens}]).to_csv(PROCESSED_FOLDER / "policy_summary_text.csv", index=False)
np.save(PROCESSED_FOLDER / "policy_summary_embedding.npy", policy_summary_embedding)
print("Saved policy_summary_text.csv and policy_summary_embedding.npy")
display(pd.read_csv(PROCESSED_FOLDER / "policy_summary_text.csv")[["topic_id", "source_run_started", "n_words", "n_tokens"]])


Saved policy_summary_text.csv and policy_summary_embedding.npy


,topic_id,source_run_started,n_words,n_tokens
0,68,2026-10-25T12:37:24+00:00,221,274
